In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.datasets import load_breast_cancer
import numpy as np

In [ ]:
def ratio(y):
        return np.mean(y)

def split_train_val_test(X, y, test_size=0.2, val_size=0.2, random_state=42): 
    if val_size <= 0:
        raise ValueError("val_size doit être > 0 pour créer un jeu de validation")

    if test_size <= 0:
        raise ValueError("test_size doit être > 0")

    if test_size + val_size >= 1:
        raise ValueError("test_size + val_size doit être < 1")
    
    X_train_val, X_test, y_train_val, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        random_state=random_state,
        stratify=y
    )

    val_ratio_adjusted = val_size / (1 - test_size)

    X_train, X_val, y_train, y_val = train_test_split(
        X_train_val,
        y_train_val,
        test_size=val_ratio_adjusted,
        random_state=random_state,
        stratify=y_train_val
    )

    print(
        f"Train : {len(X_train)} | "
        f"Validation : {len(X_val)} | "
        f"Test : {len(X_test)}"
    )

    global_ratio = ratio(y)
    print(
        "Répartition des classes conservée dans chaque jeu :",
        abs(ratio(y_train) - global_ratio) < 0.02 and
        abs(ratio(y_val) - global_ratio) < 0.02 and
        abs(ratio(y_test) - global_ratio) < 0.02
    )

    return X_train, X_val, X_test, y_train, y_val, y_test


def explorer_dataset(dataset, filter=None):
    X, y = dataset.data, dataset.target
    if filter is not None:
        class_index = y == filter
        X = X[class_index]
        y = y[class_index]
    print(f"Lignes, colonnes : {X.shape}")
    return X, y

def show_distribution(name, y):
    unique, counts = np.unique(y, return_counts=True)
    print(f"\n{name} :")
    for u, c in zip(unique, counts):
        print(f"Classe {u} : {c} ({c/len(y):.1%})")

In [ ]:
print("=====Breast Cancer Dataset=====")
breast_cancer_dataset = load_breast_cancer()
X, y = explorer_dataset(breast_cancer_dataset)
print("\n=====Phase 1 : Séparer les données proprement, train / validation / test=====")
print("\n=====Cas normal=====")
split_train_val_test(X, y)
print("\n=====Cas limite=====")
print("Tester le cas limite avec 'split_train_val_test(X, y, val_size=0)'. La fonction plante avec une gestion d'erreur")
print("\n=====Cas adversarial=====")
y_adv = np.array([0]*950 + [1]*50)
X_adv = np.random.randn(1000, 5)
unique, counts = np.unique(y_adv, return_counts=True)
print("Dataset global :")
for u, c in zip(unique, counts):
    print(f"Classe {u} : {c} ({c/len(y_adv):.1%})")
X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(X_adv, y_adv)
show_distribution("Train", y_train)
show_distribution("Validation", y_val)
show_distribution("Test", y_test)

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
def bootstrap_scores(modele, X, y, n_iterations=30, random_state=42):
    rng = np.random.default_rng(random_state)
    scores = []

    n = len(X)
    X = np.array(X)
    y = np.array(y)
    for i in range(n_iterations):

        # ---- 1. bootstrap AVEC remise ----
        idx_bootstrap = rng.choice(
            n,
            size=n,
            replace=True
        )

        # ---- 2. OOB (out of bag) ----
        oob_mask = np.ones(n, dtype=bool)
        oob_mask[idx_bootstrap] = False
        idx_oob = np.where(oob_mask)[0]

        # ---- cas OOB vide ----
        if len(idx_oob) == 0:
            continue

        X_train = X[idx_bootstrap]
        y_train = y[idx_bootstrap]

        X_test = X[idx_oob]
        y_test = y[idx_oob]

        modele.fit(X_train, y_train)

        y_pred = modele.predict(X_test)

        score = accuracy_score(y_test, y_pred)

        scores.append(score)

    if len(scores) == 0:
        print("Aucun score calculé (OOB toujours vide)")
        return []

    scores = np.array(scores)

    print(
        f"Score moyen sur {len(scores)} bootstraps : "
        f"{scores.mean():.3f} (± {scores.std():.3f})"
    )

    return scores


def bootstrap_scores_without_replace(modele, X, y, n_iterations=30, random_state=42):
    rng = np.random.default_rng(random_state)
    scores = []

    n = len(X)
    X = np.array(X)
    y = np.array(y)
    for i in range(n_iterations):
        # ---- 1. bootstrap SANS remise ----
        idx_bootstrap = rng.choice(
            n,
            size=n
        )

        # ---- 2. OOB (out of bag) ----
        oob_mask = np.ones(n, dtype=bool)
        oob_mask[idx_bootstrap] = False
        idx_oob = np.where(oob_mask)[0]

        # pour tester si une itération produit un échantillon out-ofbag vide (rare mais possible)
        # --> idx_oob = []

        X_train = X[idx_bootstrap]
        y_train = y[idx_bootstrap]

        X_test = X[idx_oob]
        y_test = y[idx_oob]

        modele.fit(X_train, y_train)

        y_pred = modele.predict(X_test)

        score = accuracy_score(y_test, y_pred)

        scores.append(score)

    if len(scores) == 0:
        print("Aucun score calculé (OOB toujours vide)")
        return []

    scores = np.array(scores)

    print(
        f"Score moyen sur {len(scores)} bootstraps : "
        f"{scores.mean():.3f} (± {scores.std():.3f})"
    )

    return scores

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier

In [ ]:
print("\n=====Phase 2 : Bootstrap et bagging, comprendre le rééchantillonnage=====")

modeles = {
    "LogisticRegression": LogisticRegression(max_iter=5000),
    "DecisionTree": DecisionTreeClassifier(),
    "RandomForest": RandomForestClassifier(n_estimators=100),
    "SVC_rbf": SVC(kernel="rbf")
}
print("\n=====Cas Happy=====")
for nom, m in modeles.items():
    print(f"{nom}: ")
    resultats = bootstrap_scores(
        m,
        X,
        y,
    )
    print("----------")

print("\n=====Cas d'oubli replace=True=====")
for nom, m in modeles.items():
    print(f"{nom}: ")
    resultats = bootstrap_scores_without_replace(
        m,
        X,
        y,
    )
    print("----------")

print("\n=====Cas n_iterations=1=====")
for nom, m in modeles.items():
    print(f"{nom}: ")
    resultats = bootstrap_scores(
        m,
        X,
        y,
        n_iterations=1
    )
    print("----------")